# Bottling Lines Quality Control

Analyse fill volumes across bottling machines. We calculate deviation from target, flag rejected bottles (those below the 2% Lower Control Limit), and visualise the distributions with boxplots, histograms, density plots, and an annotated ECDF.


In [1]:
# --- Import libraries ---
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Global settings
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [2]:
# --- Load data ---

bottling_lines = pd.read_csv("https://raw.githubusercontent.com/kostis-christodoulou/data_analytics_executives/main/data/bottling_lines.csv")


# Quick overview -- equivalent to glimpse() in R
print('Shape:', bottling_lines.shape)
print('\nColumn types:')
print(bottling_lines.dtypes)
bottling_lines.head()

Shape: (1387, 3)

Column types:
net_content    float64
machine            str
target           int64
dtype: object


,net_content,machine,target
0,762.4300,0.75 lt,750
1,749.4000,0.75 lt,750
2,737.6000,0.75 lt,750
3,752.8800,0.75 lt,750
4,755.4200,0.75 lt,750


In [ ]:
# --- Count samples per machine ---
# How many bottles did we measure from each machine?
(
    bottling_lines
    .groupby('machine', as_index=False)
    .size()
    .rename(columns={'size': 'count'})
    .sort_values('count', ascending=False)
)

In [ ]:
# --- Create delta columns ---
# delta         = absolute difference from the target fill volume
# delta_percent = relative difference (as a fraction of the target)

bottling_lines = (
    bottling_lines
    .assign(
        delta=lambda df: df['net_content'] - df['target'],
        delta_percent=lambda df: (df['net_content'] - df['target']) / df['target']
    )
)

bottling_lines[['net_content', 'target', 'delta', 'delta_percent']].describe().round(4)

In [ ]:
# --- Summary statistics per machine ---
# Equivalent to mosaic::favstats(net_content ~ machine, data=bottling_lines)

(
    bottling_lines
    .groupby('machine')['net_content']
    .agg(['count', 'mean', 'std', 'min', 'median', 'max'])
    .assign(se=lambda df: df['std'] / np.sqrt(df['count']))
    .round(3)
)

In [ ]:
# --- Flag rejected bottles ---
# LCL (Lower Control Limit) = 2% below the target fill volume
# Any bottle below LCL cannot be sold and is rejected

# We also pre-compute per-machine mean and SD for later annotations
bottling_lines = (
    bottling_lines
    .assign(
        lcl=lambda df: 0.98 * df['target'],
        rejected=lambda df: df['net_content'] < (0.98 * df['target'])
    )
    # Add machine-level mean and SD as new columns (equivalent to group_by + mutate in R)
    .pipe(lambda df: df.join(
        df.groupby('machine')['net_content']
          .agg(mean_content='mean', sd_content='std'),
        on='machine'
    ))
)

print('Columns after transformation:')
bottling_lines.dtypes

In [ ]:
# --- Count rejections per machine ---
# How many and what % of bottles fall below the LCL in each machine?

(
    bottling_lines
    .groupby(['machine', 'rejected'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
    .assign(
        percent=lambda df: df.groupby('machine')['count']
                             .transform(lambda x: x / x.sum())
    )
    .assign(percent=lambda df: df['percent'].map('{:.1%}'.format))
    .sort_values(['machine', 'rejected'])
)

In [ ]:
# --- Boxplots of net_content, faceted by machine ---
# Lets us compare spread and central tendency across machines at a glance

machines = sorted(bottling_lines['machine'].unique())
n_machines = len(machines)

fig, axes = plt.subplots(n_machines, 1, figsize=(10, 3 * n_machines), sharex=False)

for ax, machine in zip(axes, machines):
    subset = bottling_lines[bottling_lines['machine'] == machine]
    ax.boxplot(
        subset['net_content'].dropna(),
        vert=False,          # horizontal boxplot, matching R's x=net_content
        patch_artist=True,
        boxprops={'facecolor': 'steelblue', 'alpha': 0.5}
    )
    ax.set_title(machine)
    ax.set_xlabel('Net Content (ml)')
    ax.set_yticks([])

fig.suptitle('Net Content (ml) by Machine — Boxplots', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Histograms of net_content, faceted by machine ---
# Histograms show the shape of the distribution; look for bimodality or skew

g = sns.FacetGrid(
    bottling_lines,
    col='machine',
    col_wrap=2,              # 2 columns of panels
    sharey=False,
    sharex=False,
    height=3.5, aspect=1.6
)
g.map(sns.histplot, 'net_content', kde=False)
g.set_axis_labels('Net Content (ml)', 'Count')
g.set_titles('{col_name}')
g.figure.suptitle('Net Content (ml) by Machine — Histograms', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Density plots of net_content, faceted by machine ---
# Smoother version of the histogram; good for comparing shape across machines

g = sns.FacetGrid(
    bottling_lines,
    col='machine',
    col_wrap=2,
    sharey=False,
    sharex=False,
    height=3.5, aspect=1.6
)
g.map(sns.kdeplot, 'net_content', fill=True, alpha=0.4)
g.set_axis_labels('Net Content (ml)', 'Density')
g.set_titles('{col_name}')
g.figure.suptitle('Net Content (ml) by Machine — Density Plots', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- ECDF with annotated mean ± 2 SD and ± 3 SD lines ---
# This replicates the detailed stat_ecdf plot from the original R script
# Vertical lines show where ±2 SD and ±3 SD fall on the distribution

fig, axes = plt.subplots(n_machines, 1, figsize=(12, 5 * n_machines), sharex=False)

for ax, machine in zip(axes, machines):
    subset = bottling_lines[bottling_lines['machine'] == machine].dropna(subset=['net_content'])
    mu  = subset['mean_content'].iloc[0]
    sig = subset['sd_content'].iloc[0]

    # Draw ECDF line
    sns.ecdfplot(data=subset, x='net_content', ax=ax)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))

    # --- Annotate key reference lines ---
    for n_sd, colour in [(2, 'orange'), (3, 'tomato')]:
        for sign, label_prefix in [(-1, '-'), (1, '+')]:
            xval  = mu + sign * n_sd * sig
            label = f'Mean {label_prefix} {n_sd}×SD = {xval:.2f}'
            ax.axvline(xval, color=colour, linestyle='--', linewidth=1.2)
            ax.text(xval, 0.5, f'\n{label}',
                    rotation=90, va='center', ha='right',
                    color=colour, fontsize=7)

    # Mean line
    ax.axvline(mu, color='steelblue', linewidth=1.5)
    ax.text(mu, 0.5, f'\nMean = {mu:.2f}',
            rotation=90, va='center', ha='right',
            color='steelblue', fontsize=7)

    ax.set_title(machine)
    ax.set_xlabel('Net Content (ml)')
    ax.set_ylabel('')

fig.suptitle('ECDF of Net Content with Mean ± 2/3 SD Annotations', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Jitter plot with 95% CI for the mean ---
# Shows individual data points and a big orange dot = 95% CI for the mean
# Are machines biased toward overfilling? (desirable to avoid waste) or underfilling?

fig, ax = plt.subplots(figsize=(10, 6))

# Jitter strip plot
sns.stripplot(
    data=bottling_lines,
    x='net_content',
    y='machine',
    alpha=0.35,
    size=3,
    jitter=True,
    ax=ax
)

# Overlay the 95% CI for the mean (equivalent to stat_summary with fun.data='mean_se')
for i, machine in enumerate(bottling_lines['machine'].unique()):
    vals = bottling_lines.loc[bottling_lines['machine'] == machine, 'net_content'].dropna()
    n    = len(vals)
    mu   = vals.mean()
    se   = vals.std(ddof=1) / np.sqrt(n)
    ci   = 1.96 * se
    ax.errorbar(
        x=mu, y=i,
        xerr=ci,
        fmt='o',
        color='orange',
        markersize=10,
        linewidth=2,
        zorder=5,
        label='95% CI for mean' if i == 0 else ''
    )

ax.set_xlabel('Net Content (ml)')
ax.set_ylabel('')
ax.set_title('Mean Content by Machine — Bias to Overfilling?')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()